In [ ]:
from libraries import *
from parameters import *
import pegasusio as io

In [ ]:
%load_ext rpy2.ipython

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
conf_mt_prefix = 'MT-' 

In [ ]:
adata = sc.read("./outputs/anndata/adataNeuro.h5ad")

In [ ]:
adata.var

In [ ]:
adata.layers['counts'] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=par_preprocessing_target_sum)
sc.pp.log1p(adata)
adata.raw = adata

In [ ]:
gene_list_url = 'https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt'
cell_cycle_genes = [str(x.strip(), 'utf-8').upper() for x in urlopen(gene_list_url)] # capitalize = shame
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]


sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)


In [ ]:
sc.pp.regress_out(adata, ['log10_n_umis', 'mt_frac', 'n_genes'], n_jobs=30)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=par_downstream_n_top_genes)
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10)
sc.tl.umap(adata)

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (8, 3)}):
    sc.pl.violin(adata, ['mt_frac'], 
                stripplot=False, inner='box',
                 groupby='Sample_type')

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (12, 3)}):
    sc.pl.violin(adata, ['n_genes'], 
                stripplot=False, inner='box',
                 groupby='Sample_type')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins


In [ ]:
sc.tl.leiden(adata, resolution=0.5)

f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='leiden', legend_loc='on data', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=8);

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='mt_frac', legend_loc='on data', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=8);

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='phase', legend_loc='on data', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=8);

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='n_genes', legend_loc='on data', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=8);

In [ ]:
adata.obs["Sample_type"] = adata.obs["Sample_type"].astype("category")

In [ ]:
f, ax = plt.subplots(figsize=(6, 6))

sc.pl.umap(adata, color='Sample_type', 
           legend_fontoutline=3, legend_fontsize=14, ax=ax, 
           legend_fontweight='normal', title='Clusters', size=18);

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", n_genes=2000, method="t-test_overestim_var")
sc.tl.dendrogram(adata, groupby='leiden')
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=15, standard_scale='var', cmap='Blues')

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="Sample_type", n_genes=2000, method="t-test_overestim_var")
sc.tl.dendrogram(adata, groupby='Sample_type')
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=40, standard_scale='var', cmap='Blues')

In [ ]:
markerGenes = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
markerGenes = markerGenes.iloc[0:40,:]
markerGenes.to_csv("./TextFiles/NEPC_adenocarcinoma_markerGenes.csv")

In [ ]:
markerGenes

In [ ]:
for i in  markerGenes.columns:
    print(i)
    myGeneList = [x for x in markerGenes.loc[:,i] if x != ' ']
    myGeneList = [x for x in myGeneList if x !="nan"]
    myGeneList = [x for x in myGeneList if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]
    
    myGeneList = [x for x in myGeneList if x in adata.var_names]
    
    sc.pl.heatmap(adata, myGeneList, groupby='Sample_type', dendrogram=True)
    sc.pl.dotplot(adata, myGeneList, groupby='Sample_type', dendrogram=True)

    #sc.pl.violin(adata, myGeneList, groupby="Sample_type")
    
#     f, ax = plt.subplots(figsize=(16, 16))
#     sc.pl.stacked_violin(adata, myGeneList, groupby='Sample_type', dendrogram=True, ax=ax, size=10)


    # print(myGeneList)
    # print(sum(pd.Series(myGeneList).isin(adata.var_names)))
    # if(sum(pd.Series(myGeneList).isin(adata.var_names)) > 1):
    #     sc.tl.score_genes(adata=adata, gene_list=myGeneList, score_name=i)
    #     sc.pl.umap(adata, color=i, size=10, color_map="coolwarm", vmax=0.3, vmin=-0.3)
    #     f, ax = plt.subplots(figsize=(12, 4))
    #     sc.pl.violin(adata, i, groupby='Sample_type', ax=ax)

In [ ]:
geneSignatures = pd.DataFrame(pd.read_csv("./TextFiles/Human_NE_signatures.csv"))

In [ ]:
geneSignatures

In [ ]:
for i in  geneSignatures.columns:
    print(i)
    myGeneList = [x for x in geneSignatures.loc[:,i] if x != ' ']
    myGeneList = [x for x in myGeneList if x !="nan"]
    myGeneList = [x for x in myGeneList if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]
    
    myGeneList = [x for x in myGeneList if x in adata.var_names]
    
    sc.pl.heatmap(adata, myGeneList, groupby='Sample_type', dendrogram=True, figsize=(8,4))
    sc.pl.dotplot(adata, myGeneList, groupby='Sample_type', dendrogram=True)

    #sc.pl.violin(adata, myGeneList, groupby="Sample_type")
    
#     f, ax = plt.subplots(figsize=(16, 16))
#     sc.pl.stacked_violin(adata, myGeneList, groupby='Sample_type', dendrogram=True, ax=ax, size=10)


    # print(myGeneList)
    # print(sum(pd.Series(myGeneList).isin(adata.var_names)))
    # if(sum(pd.Series(myGeneList).isin(adata.var_names)) > 1):
    #     sc.tl.score_genes(adata=adata, gene_list=myGeneList, score_name=i)
    #     sc.pl.umap(adata, color=i, size=10, color_map="coolwarm", vmax=0.3, vmin=-0.3)
    #     f, ax = plt.subplots(figsize=(12, 4))
    #     sc.pl.violin(adata, i, groupby='Sample_type', ax=ax)

In [ ]:
enDistMat = pd.DataFrame(np.zeros(shape=(len(adata.obs["Sample_type"].unique()), len(adata.obs["Sample_type"].unique()))))
cond = list(adata.obs["Sample_type"].unique())
cond.sort()
enDistMat.index = cond
enDistMat.columns = cond


In [ ]:
from geomloss import SamplesLoss
import torch
Loss =  SamplesLoss("energy")

for i in cond:
    ad_rna_i = adata[adata.obs["Sample_type"]==i,:]
    for j in cond:
        ad_rna_j = adata[adata.obs["Sample_type"]==j,:]
        
        enDistMat.loc[i,j] = Loss( torch.from_numpy(ad_rna_i.obsm["X_pca"]), torch.from_numpy(ad_rna_j.obsm["X_pca"]) ).item()       

In [ ]:
%%R -i enDistMat -w 6 -h 6 -u in

library(pheatmap)

pheatmap(enDistMat, method="ward.D", cluster_rows= TRUE, cluster_cols=TRUE)